# Pruebas del helper de transformación de coordenadas proyectadas a lat/lon

Este notebook prueba el helper espacial para convertir coordenadas proyectadas X/Y a latitud/longitud. Se usan casos representativos de EOD y ADATRAP para validar transformación, parseo de valores y manejo de datos faltantes.

In [1]:
import pandas as pd
from pylondrina.transforms.spatial import project_xy_to_latlon

### Caso A - EOD: origen y destino

In [8]:
df_eod = pd.DataFrame({
    "OrigenCoordX": [345000.0, 346200.5],
    "OrigenCoordY": [6291000.0, 6292200.0],
    "DestinoCoordX": [348000.0, 349150.0],
    "DestinoCoordY": [6293000.0, 6294100.0],
})

df_eod_1 = project_xy_to_latlon(
    df_eod,
    x_col="OrigenCoordX",
    y_col="OrigenCoordY",
    source_crs="EPSG:5361",
    lon_col="OrigenCoordLon",
    lat_col="OrigenCoordLat",
    keep_debug_cols=False,
    drop_input_cols=True,
)

df_eod_1 = project_xy_to_latlon(
    df_eod_1,
    x_col="DestinoCoordX",
    y_col="DestinoCoordY",
    source_crs="EPSG:5361",
    lon_col="DestinoCoordLon",
    lat_col="DestinoCoordLat",
    keep_debug_cols=False,
    drop_input_cols=True,
)

display(df_eod_1)

,OrigenCoordLon,OrigenCoordLat,DestinoCoordLon,DestinoCoordLat
0,-70.668810,-33.509330,-70.636182,-33.491729
1,-70.655683,-33.498684,-70.623623,-33.481974


### Caso B - ADATRAP: después del merge con tabla de estaciones

In [9]:
df_adatrap = pd.DataFrame({
    "subida_x": ["349000", "350200"],
    "subida_y": ["6294500", "6295100"],
    "bajada_x": ["351000", "352100"],
    "bajada_y": ["6296000", "6297000"],
})

df_adatrap = project_xy_to_latlon(
    df_adatrap,
    x_col="subida_x",
    y_col="subida_y",
    source_crs="EPSG:5361",
    lon_col="subida_lon",
    lat_col="subida_lat",
    keep_debug_cols=False,
    drop_input_cols=False,
)

df_adatrap = project_xy_to_latlon(
    df_adatrap,
    x_col="bajada_x",
    y_col="bajada_y",
    source_crs="EPSG:5361",
    lon_col="bajada_lon",
    lat_col="bajada_lat",
    keep_debug_cols=False,
    drop_input_cols=False,
)

display(df_adatrap)

,subida_x,subida_y,bajada_x,bajada_y,subida_lon,subida_lat,bajada_lon,bajada_lat
0,349000,6294500,351000,6296000,-70.625169,-33.478346,-70.603402,-33.465102
1,350200,6295100,352100,6297000,-70.612158,-33.473105,-70.591404,-33.456239


### Caso C - depuración útil con strings raros, vacíos o ceros

Este sirve para verificar que realmente está manejando casos problemáticos.

In [11]:
df_debug = pd.DataFrame({
    "x": ["349000", "349100,5", "", None, "abc", "0"],
    "y": ["6294500", "6294600,5", "6294700", None, "xyz", "0"],
})

df_debug_out = project_xy_to_latlon(
    df_debug,
    x_col="x",
    y_col="y",
    source_crs="EPSG:5361",
    lon_col="lon",
    lat_col="lat",
    decimal_comma=True,
    zero_as_missing=True,
    keep_debug_cols=True,
    drop_input_cols=False,
)

display(df_debug_out)

,x,y,lon,lat,__x_parsed,__y_parsed,__x_status,__y_status,__lon_latlon_status
0,349000,6294500,-70.625169,-33.478346,349000.0,6294500.0,ok_string,ok_string,transformed
1,"349100,5","6294600,5",-70.624071,-33.477454,349100.5,6294600.5,ok_string,ok_string,transformed
2,,6294700,NaN,NaN,NaN,6294700.0,empty,ok_string,not_transformed
3,None,None,NaN,NaN,NaN,NaN,null,null,not_transformed
4,abc,xyz,NaN,NaN,NaN,NaN,non_numeric,non_numeric,not_transformed
5,0,0,NaN,NaN,NaN,NaN,zero_as_missing,zero_as_missing,not_transformed
